# What is State:

The shared information travelling throughout the graph i.e.,from one node to the other node.

# 1.The format to store the chat history

In [ ]:
!pip install -q -U langchain-openai

In [3]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key:")

messages=[HumanMessage(content=("My payment was deducted but my order is still pending.")),
        AIMessage(content=("I'll help you check your payment and Order status.")),
         HumanMessage(content=("My order id is ORD12345."))]

print("=============MESSAGE HISTORY======================\n")

for message in messages:
    print("TYPE:",type(message).__name__)
    print("CONTENT:",message.content)
    print("------------------------------")

Enter your OpenAI API key: ········


=============MESSAGE HISTORY======================

TYPE: HumanMessage
CONTENT: My payment was deducted but my order is still pending.
------------------------------
TYPE: AIMessage
CONTENT: I'll help you check your payment and Order status.
------------------------------
TYPE: HumanMessage
CONTENT: My order id is ORD12345.
------------------------------


In [5]:
#Initialize chat model

model = ChatOpenAI(model="gpt-4o-mini",temperature=0)
response=model.invoke(messages)

print("\n==================AI RESPONSE==========================")
print("TYPE:",type(response).__name__)
print("\nCONTENT:")
print(response.content)

#Display response METADATA
print("\n==========RESPONSE METADATA================")
print(response.response_metadata)

#Display token usage
print("\n========TOKEN USAGE=================\n")
print(response.usage_metadata)


==================AI RESPONSE==========================
TYPE: AIMessage

CONTENT:
I don't have access to specific order systems or databases to check the status of your order. However, I can suggest some steps you can take:

1. **Check Your Email**: Look for any confirmation emails regarding your order. Sometimes, there may be updates or additional information.

2. **Contact Customer Support**: Reach out to the customer service team of the company you ordered from. Provide them with your order ID (ORD12345) and details about your payment.

3. **Check Payment Method**: Verify that the payment was successfully processed through your bank or payment service. Sometimes, there can be delays in processing.

4. **Review Order Status Online**: If the company has an online portal, log in to your account to check the status of your order.

5. **Wait for a Response**: If you’ve already contacted customer support, give them some time to respond, as they may be experiencing high volumes of inquiri

# Point 2: Hwo to build a Tool Calling LLM using LangGraph?

In [6]:
!pip install -q -U langchain-openai langchain-core


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.tools import tool

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"]= getpass("Enter your OpenAI API Key:")

@tool
def check_payment_status(order_id: str) -> str:
    """
    Check the payment status for a customer order.

    Args:
    order_id: Customer order ID.
    """
    if order_id=="ORD12345":
        return ("Order is currently pending.")

    return("Order information not found")

@tool
def check_order_status(order_id: str) ->str:
    """
    Check the current order status.

    Args:
    order_id = Customer order ID.
    """
    if order_id=="ORD12345":
        return ("Order is currently pending.")
        
    return ("Order information not found.")

@tool
def create_support_ticket(issue: str)->str:
    """
    Create a support ticket for a customer issue.

    Args:
    issue:Description of the issue.
    """
    return (f"Support ticket created for {issue}")

tools=[check_payment_status,check_order_status,create_support_ticket]

In [12]:
# Initialize the Model

model=ChatOpenAI(model="gpt-4o-mini",temperature=0)

#Bind Tools
model_with_tools=model.bind_tools(tools)

#User Question
messages=[HumanMessage(content=("My payment was deducted"
                                "But my order is still pending."
                                "My order ID is ORD12345."
                                "Please check my payment status."))]

response = model_with_tools.invoke(messages)

print("\n=============MODEL RESPONSE ====================\n")
print(response)

#Display tool call
print("\n============Tool Calls===================\n")
print(response.tool_calls)




=============MODEL RESPONSE ====================

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 144, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_894e7baa82', 'id': 'chatcmpl-ERLohq7DEIeWxLldoOwZd2qhcrElj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0cf7c-5443-7d20-b15b-726a665b8d4d-0' tool_calls=[{'name': 'check_payment_status', 'args': {'order_id': 'ORD12345'}, 'id': 'call_FDLfsU2TDDxU87UnqRCb59vl', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 144, 'output_tokens': 18